# Trajectory Analysis Loader

Set `TRAJECTORY_DIR` to any run folder under `trajectories/`, then run the notebook. It loads every trajectory JSON into raw dictionaries for deeper analysis and builds a compact table for quick inspection.

Main outputs:
- `records`: list of metadata-wrapped raw JSON entries
- `raw_data`: nested dict as `raw_data[method][index][sample] -> raw trajectory dict`
- `by_method`: dict as `by_method[method] -> list[entry]`
- `by_problem`: dict as `by_problem[index][method][sample] -> raw trajectory dict`
- `by_key`: dict as `by_key[(method, index, sample)] -> raw trajectory dict`
- `records_df` and `summary_df`: pandas tables, when pandas is available

In [53]:
from collections import defaultdict
from pathlib import Path
from pprint import pprint
import json
import re

try:
    import pandas as pd
except ImportError:
    pd = None

# Change this to the run folder you want to inspect.
TRAJECTORY_DIR = Path("trajectories/test-llama-bm-617-cache_llama_bigbench_movie_s2")

trajectory_dir_nocache = Path("trajectories/test-llama-bm-617-nocache_llama_bigbench_movie_s2")

METHODS = ['vanilla', 'rejection', 'lawyer', 'stepbootstrap']
method = METHODS[3]
    

TRAJ_FILE_RE = re.compile(
    r"^traj_(?P<index>\d+)(?:_sample_(?P<sample>\d+))?(?P<error>_error)?\.json$"
)


def freeze_defaultdict(value):
    """Convert nested defaultdicts into normal dicts for easier notebook use."""
    if isinstance(value, defaultdict):
        value = dict(value)
    if isinstance(value, dict):
        return {key: freeze_defaultdict(item) for key, item in value.items()}
    if isinstance(value, list):
        return [freeze_defaultdict(item) for item in value]
    return value


def parse_trajectory_filename(path):
    match = TRAJ_FILE_RE.match(path.name)
    if match is None:
        return {"index": None, "sample": None, "is_error": False}

    sample = match.group("sample")
    return {
        "index": int(match.group("index")),
        "sample": int(sample) if sample is not None else None,
        "is_error": bool(match.group("error")),
    }


def method_for(path, root):
    relative = path.relative_to(root)
    return relative.parts[0] if len(relative.parts) > 1 else "root"


def load_trajectory_folder(folder):
    root = Path(folder)
    if not root.exists():
        raise FileNotFoundError(f"Trajectory folder does not exist: {root}")
    if not root.is_dir():
        raise NotADirectoryError(f"Expected a directory: {root}")

    records = []
    errors = []

    for path in sorted(root.rglob("*.json")):
        with path.open("r", encoding="utf-8") as handle:
            data = json.load(handle)

        meta = parse_trajectory_filename(path)
        entry = {
            **meta,
            "method": method_for(path, root),
            "path": path,
            "relative_path": path.relative_to(root),
            "data": data,
        }
        records.append(entry)

        if meta["is_error"] or "error_type" in data:
            errors.append(entry)

    by_method = defaultdict(list)
    raw_data = defaultdict(lambda: defaultdict(dict))
    by_problem = defaultdict(lambda: defaultdict(dict))
    by_key = {}

    for entry in records:
        method = entry["method"]
        index = entry["index"]
        sample = entry["sample"]
        data = entry["data"]

        by_method[method].append(entry)
        raw_data[method][index][sample] = data
        by_problem[index][method][sample] = data
        by_key[(method, index, sample)] = data

    return {
        "root": root,
        "records": records,
        "errors": errors,
        "by_method": freeze_defaultdict(by_method),
        "raw_data": freeze_defaultdict(raw_data),
        "by_problem": freeze_defaultdict(by_problem),
        "by_key": by_key,
    }

In [54]:
loaded = load_trajectory_folder(TRAJECTORY_DIR)

records = loaded["records"]
errors = loaded["errors"]
by_method = loaded["by_method"]
raw_data = loaded["raw_data"]
by_problem = loaded["by_problem"]
by_key = loaded["by_key"]

print(f"Loaded {len(records)} JSON files from {loaded['root']}")
print(f"Errors: {len(errors)}")
print("\nFiles by method:")
for method, entries in sorted(by_method.items()):
    print(f"  {method}: {len(entries)}")


loaded_nocache = load_trajectory_folder(trajectory_dir_nocache)
records_nocache = loaded_nocache["records"]
errors_nocache = loaded_nocache["errors"]
by_method_nocache = loaded_nocache["by_method"]
raw_data_nocache = loaded_nocache["raw_data"]
by_problem_nocache = loaded_nocache["by_problem"]
by_key_nocache = loaded_nocache["by_key"]

print(f"Loaded {len(records_nocache)} JSON files from {loaded_nocache['root']}")
print(f"Errors: {len(errors_nocache)}")
print("\nFiles by method:")
for method, entries in sorted(by_method_nocache.items()):
    print(f"  {method}: {len(entries)}")

Loaded 210 JSON files from trajectories/test-llama-bm-617-cache_llama_bigbench_movie_s2
Errors: 0

Files by method:
  lawyer: 4
  rejection: 4
  stepbootstrap: 200
  vanilla: 2
Loaded 210 JSON files from trajectories/test-llama-bm-617-nocache_llama_bigbench_movie_s2
Errors: 0

Files by method:
  lawyer: 4
  rejection: 4
  stepbootstrap: 200
  vanilla: 2


In [56]:
all_total_confidence_times = [traj['data']['timings']['total_confidence_time'] for traj in by_method[method] ]
all_total_confidence_times_nocache = [traj['data']['timings']['total_confidence_time'] for traj in by_method_nocache[method] ]
all_answer_prob_times = [traj['data']['timings']['confidence_time']['answer_prob_time'] for traj in by_method[method] ]
all_answer_ent_times = [traj['data']['timings']['confidence_time']['answer_ent_time'] for traj in by_method[method] ]
all_indirect_times = [traj['data']['timings']['confidence_time']['indirect_time'] for traj in by_method[method] ]
all_verbconf_times = [traj['data']['timings']['confidence_time']['verbconf_time'] for traj in by_method[method] ]
all_answer_prob_times_nocache = [traj['data']['timings']['confidence_time']['answer_prob_time'] for traj in by_method_nocache[method] ]
all_answer_ent_times_nocache = [traj['data']['timings']['confidence_time']['answer_ent_time'] for traj in by_method_nocache[method] ]
all_indirect_times_nocache = [traj['data']['timings']['confidence_time']['indirect_time'] for traj in by_method_nocache[method] ]
all_verbconf_times_nocache = [traj['data']['timings']['confidence_time']['verbconf_time'] for traj in by_method_nocache[method] ]
all_answer_prob_times_nocache = [traj['data']['timings']['confidence_time']['answer_prob_time'] for traj in by_method_nocache[method] ]
all_answer_ent_times_nocache = [traj['data']['timings']['confidence_time']['answer_ent_time'] for traj in by_method_nocache[method] ]
all_indirect_times_nocache = [traj['data']['timings']['confidence_time']['indirect_time'] for traj in by_method_nocache[method] ]

In [58]:
all_answer_prob_times

[0.014917135238647461,
 3.743171691894531e-05,
 3.647804260253906e-05,
 3.552436828613281e-05,
 3.6716461181640625e-05,
 3.695487976074219e-05,
 3.504753112792969e-05,
 3.4809112548828125e-05,
 3.62396240234375e-05,
 3.647804260253906e-05,
 3.552436828613281e-05,
 3.7670135498046875e-05,
 3.4332275390625e-05,
 3.647804260253906e-05,
 3.600120544433594e-05,
 3.600120544433594e-05,
 3.504753112792969e-05,
 3.838539123535156e-05,
 3.6716461181640625e-05,
 3.528594970703125e-05,
 3.4809112548828125e-05,
 3.457069396972656e-05,
 3.552436828613281e-05,
 3.5762786865234375e-05,
 4.267692565917969e-05,
 3.409385681152344e-05,
 3.457069396972656e-05,
 3.4809112548828125e-05,
 3.647804260253906e-05,
 3.457069396972656e-05,
 3.528594970703125e-05,
 4.100799560546875e-05,
 3.528594970703125e-05,
 3.552436828613281e-05,
 3.409385681152344e-05,
 3.719329833984375e-05,
 3.62396240234375e-05,
 3.552436828613281e-05,
 3.504753112792969e-05,
 3.695487976074219e-05,
 3.409385681152344e-05,
 3.50475311279

In [59]:
all_answer_prob_times_nocache

[5.53131103515625e-05,
 3.504753112792969e-05,
 3.552436828613281e-05,
 3.528594970703125e-05,
 3.504753112792969e-05,
 3.552436828613281e-05,
 3.4809112548828125e-05,
 3.5762786865234375e-05,
 3.457069396972656e-05,
 3.2901763916015625e-05,
 3.266334533691406e-05,
 3.528594970703125e-05,
 3.647804260253906e-05,
 3.457069396972656e-05,
 3.5762786865234375e-05,
 3.7670135498046875e-05,
 3.600120544433594e-05,
 3.528594970703125e-05,
 3.3855438232421875e-05,
 3.695487976074219e-05,
 3.3855438232421875e-05,
 3.337860107421875e-05,
 3.457069396972656e-05,
 3.3855438232421875e-05,
 3.4332275390625e-05,
 3.504753112792969e-05,
 3.600120544433594e-05,
 3.600120544433594e-05,
 3.528594970703125e-05,
 3.361701965332031e-05,
 3.552436828613281e-05,
 3.504753112792969e-05,
 3.4809112548828125e-05,
 3.62396240234375e-05,
 3.457069396972656e-05,
 3.314018249511719e-05,
 3.528594970703125e-05,
 3.4809112548828125e-05,
 3.457069396972656e-05,
 3.528594970703125e-05,
 3.4332275390625e-05,
 3.480911254